# ORACLE ZERO
## Uncertainty-Aware Machine Learning Reliability and Selective Prediction

**Goal:** Train and calibrate a classifier, flag unfamiliar inputs, and test a policy that abstains when a prediction may be unreliable.

This runnable prototype includes data inspection, preprocessing, calibration, novelty detection, selective prediction, risk–coverage analysis, a controlled distribution-shift experiment, visualizations, and an audit report.

**Important:** Educational research prototype only—not a clinical or safety-critical decision system. Novelty scores and confidence are warning signals, not guarantees of correctness.

## 1. Dependencies
If needed, uncomment the install command.

In [ ]:
# %pip install -U numpy pandas matplotlib scikit-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, classification_report
)
from sklearn.base import clone

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports successful.")

## 2. Load and inspect data
The built-in Wisconsin Diagnostic Breast Cancer dataset is used only to demonstrate the methodology; it is not intended for clinical use.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data.copy()
y = dataset.target.copy()
print(f"Rows: {len(X):,} | Features: {X.shape[1]}")
print("Target mapping:", dict(enumerate(dataset.target_names)))
display(X.head())
display(y.value_counts().rename(index=dict(enumerate(dataset.target_names))).to_frame("count"))
summary = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing_count": X.isna().sum(),
    "unique_values": X.nunique(),
    "mean": X.mean(),
    "std": X.std()
})
display(summary.head(15))
print("Duplicate feature rows:", int(X.duplicated().sum()))

## 3. Split before fitting preprocessing
Validation data selects the abstention threshold; the test set is reserved for final evaluation.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)
print("Train:", X_train.shape, "| Validation:", X_val.shape, "| Test:", X_test.shape)

## 4. Train baseline and calibrated models
Preprocessing is fitted only on training data. Probability calibration uses cross-validation within the training set.

In [ ]:
base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])
base_model.fit(X_train, y_train)

calibrated_model = CalibratedClassifierCV(
    estimator=clone(base_model), method="sigmoid", cv=5
)
calibrated_model.fit(X_train, y_train)
print("Models trained.")

## 5. Evaluation helpers

In [ ]:
def expected_calibration_error(y_true, probabilities, n_bins=10):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)
    confidence = probabilities.max(axis=1)
    predicted = probabilities.argmax(axis=1)
    correct = (predicted == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        if i == n_bins - 1:
            mask = (confidence >= edges[i]) & (confidence <= edges[i + 1])
        else:
            mask = (confidence >= edges[i]) & (confidence < edges[i + 1])
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)

def evaluate_predictions(y_true, probabilities, name):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)
    pred = probabilities.argmax(axis=1)
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, pred),
        "precision_macro": precision_score(y_true, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, pred, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities[:, 1]),
        "brier_score_class_1": brier_score_loss(y_true, probabilities[:, 1]),
        "mean_confidence": probabilities.max(axis=1).mean(),
        "ece_approx": expected_calibration_error(y_true, probabilities)
    }

## 6. Compare models on validation data

In [ ]:
val_probs_base = base_model.predict_proba(X_val)
val_probs_cal = calibrated_model.predict_proba(X_val)
val_metrics = pd.DataFrame([
    evaluate_predictions(y_val, val_probs_base, "Baseline logistic regression"),
    evaluate_predictions(y_val, val_probs_cal, "Calibrated logistic regression")
]).set_index("model")
display(val_metrics.round(4))

## 7. Calibration plot
The plot compares predicted probabilities for class 1 with observed frequencies. The diagonal indicates ideal calibration.

In [ ]:
plt.figure(figsize=(7, 6))
for name, probs in [("Baseline", val_probs_base), ("Calibrated", val_probs_cal)]:
    observed, predicted = calibration_curve(y_val, probs[:, 1], n_bins=10, strategy="quantile")
    plt.plot(predicted, observed, marker="o", label=name)
plt.plot([0, 1], [0, 1], "--", label="Ideal calibration")
plt.xlabel("Mean predicted probability for class 1")
plt.ylabel("Observed fraction of class 1")
plt.title("Validation Calibration Curves")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Unfamiliar-input detector
Isolation Forest is fitted on training features only. Its novelty score is a heuristic warning, not proof of an error or distribution shift.

In [ ]:
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled = preprocessor.transform(X_val)
X_test_scaled = preprocessor.transform(X_test)

ood_detector = IsolationForest(n_estimators=300, contamination="auto", random_state=RANDOM_STATE)
ood_detector.fit(X_train_scaled)
val_novelty = -ood_detector.score_samples(X_val_scaled)
test_novelty = -ood_detector.score_samples(X_test_scaled)

# Flag the most unusual 10% of validation examples as a demonstration policy.
novelty_threshold = float(np.quantile(val_novelty, 0.90))
print(f"Novelty threshold: {novelty_threshold:.4f}")
print(f"Validation flagged: {(val_novelty > novelty_threshold).mean():.1%}")
print(f"Test flagged: {(test_novelty > novelty_threshold).mean():.1%}")

## 9. Selective prediction
The system abstains when calibrated confidence is below a threshold or the novelty score exceeds its threshold. Choose confidence threshold using validation data only.

In [ ]:
def selective_metrics(y_true, probabilities, novelty_scores, confidence_threshold, novelty_cutoff):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)
    confidence = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    accepted = (confidence >= confidence_threshold) & (novelty_scores <= novelty_cutoff)
    coverage = accepted.mean()
    if accepted.any():
        acc = accuracy_score(y_true[accepted], predictions[accepted])
        f1 = f1_score(y_true[accepted], predictions[accepted], average="macro", zero_division=0)
    else:
        acc, f1 = np.nan, np.nan
    return {
        "confidence_threshold": float(confidence_threshold),
        "coverage": float(coverage),
        "accepted_count": int(accepted.sum()),
        "abstained_count": int((~accepted).sum()),
        "selective_accuracy": float(acc) if not np.isnan(acc) else np.nan,
        "selective_error_rate": float(1-acc) if not np.isnan(acc) else np.nan,
        "selective_f1_macro": float(f1) if not np.isnan(f1) else np.nan
    }, accepted

candidate_rows = []
candidate_thresholds = np.unique(np.quantile(val_probs_cal.max(axis=1), np.linspace(0, 0.80, 41)))
for threshold in candidate_thresholds:
    metrics, _ = selective_metrics(y_val, val_probs_cal, val_novelty, threshold, novelty_threshold)
    if metrics["coverage"] >= 0.70 and metrics["accepted_count"] > 0:
        candidate_rows.append(metrics)

if candidate_rows:
    candidates = pd.DataFrame(candidate_rows)
    best = candidates.sort_values(["selective_accuracy", "coverage"], ascending=[False, False]).iloc[0]
    confidence_threshold = float(best["confidence_threshold"])
else:
    confidence_threshold = 0.0

val_selective, val_accepted = selective_metrics(
    y_val, val_probs_cal, val_novelty, confidence_threshold, novelty_threshold
)
print("Selected confidence threshold:", round(confidence_threshold, 4))
display(pd.DataFrame([val_selective]).round(4))

## 10. Risk–coverage curve
This visualizes the error rate among accepted predictions as coverage changes.

In [ ]:
rows = []
for threshold in np.linspace(0, 0.99, 60):
    m, _ = selective_metrics(y_val, val_probs_cal, val_novelty, threshold, novelty_threshold)
    if m["accepted_count"] > 0:
        rows.append({"threshold": threshold, "coverage": m["coverage"], "risk": m["selective_error_rate"]})
rc = pd.DataFrame(rows).sort_values("coverage")
plt.figure(figsize=(8, 5))
plt.plot(rc["coverage"], rc["risk"], marker=".", linewidth=1.5)
plt.xlabel("Coverage (fraction of predictions accepted)")
plt.ylabel("Error rate among accepted predictions")
plt.title("Validation Risk–Coverage Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Controlled distribution-shift stress test
Perturb selected standardized features in the test set. This is a synthetic experiment, not a realistic model of all real-world shifts.

In [ ]:
X_test_scaled_shifted = X_test_scaled.copy()
rng = np.random.default_rng(RANDOM_STATE)
n_shift = max(1, X_test_scaled_shifted.shape[1] // 5)
shifted_columns = rng.choice(X_test_scaled_shifted.shape[1], size=n_shift, replace=False)
X_test_scaled_shifted[:, shifted_columns] += rng.normal(
    loc=2.0, scale=0.5, size=(len(X_test_scaled_shifted), n_shift)
)

# Invert scaling to create shifted data in the original feature units.
scaler = preprocessor.named_steps["scaler"]
X_test_shifted = pd.DataFrame(
    scaler.inverse_transform(X_test_scaled_shifted), columns=X.columns, index=X_test.index
)
shifted_probs = calibrated_model.predict_proba(X_test_shifted)
shifted_novelty = -ood_detector.score_samples(X_test_scaled_shifted)

original_probs = calibrated_model.predict_proba(X_test)
stress_results = pd.DataFrame([
    {"scenario": "Original held-out test",
     "accuracy": accuracy_score(y_test, original_probs.argmax(axis=1)),
     "mean_confidence": original_probs.max(axis=1).mean(),
     "novelty_flag_rate": (test_novelty > novelty_threshold).mean()},
    {"scenario": "Synthetic shifted test",
     "accuracy": accuracy_score(y_test, shifted_probs.argmax(axis=1)),
     "mean_confidence": shifted_probs.max(axis=1).mean(),
     "novelty_flag_rate": (shifted_novelty > novelty_threshold).mean()}
])
display(stress_results.round(4))
print("Shifted feature indices:", sorted(shifted_columns.tolist()))

## 12. Final held-out test evaluation
Thresholds were chosen using validation data. The held-out test set is used here for final evaluation.

In [ ]:
test_probs_base = base_model.predict_proba(X_test)
test_probs_cal = calibrated_model.predict_proba(X_test)
test_metrics = pd.DataFrame([
    evaluate_predictions(y_test, test_probs_base, "Baseline logistic regression"),
    evaluate_predictions(y_test, test_probs_cal, "Calibrated logistic regression")
]).set_index("model")
display(test_metrics.round(4))

test_selective, test_accepted = selective_metrics(
    y_test, test_probs_cal, test_novelty, confidence_threshold, novelty_threshold
)
display(pd.DataFrame([test_selective]).round(4))
print(classification_report(
    y_test, test_probs_cal.argmax(axis=1), target_names=dataset.target_names
))

## 13. Inspect individual prediction decisions

In [ ]:
test_decisions = pd.DataFrame({
    "actual_label": pd.Series(y_test, index=y_test.index).map(dict(enumerate(dataset.target_names))),
    "predicted_label": pd.Series(test_probs_cal.argmax(axis=1), index=y_test.index).map(dict(enumerate(dataset.target_names))),
    "confidence": test_probs_cal.max(axis=1),
    "probability_class_1": test_probs_cal[:, 1],
    "novelty_score": test_novelty,
    "confidence_pass": test_probs_cal.max(axis=1) >= confidence_threshold,
    "novelty_pass": test_novelty <= novelty_threshold,
    "decision": np.where(test_accepted, "PREDICTION AVAILABLE", "ABSTAIN / REVIEW"),
    "correct": test_probs_cal.argmax(axis=1) == y_test.to_numpy()
}, index=y_test.index)
display(test_decisions.head(20))
print(test_decisions["decision"].value_counts())

## 14. Export audit report and test decisions

In [ ]:
report_lines = [
    "ORACLE ZERO — Machine Learning Reliability Audit",
    "=" * 52,
    "Dataset: scikit-learn Wisconsin Diagnostic Breast Cancer (educational demonstration only)",
    f"Rows: {len(X)} | Features: {X.shape[1]}",
    f"Train/validation/test sizes: {len(X_train)}/{len(X_val)}/{len(X_test)}",
    f"Confidence threshold: {confidence_threshold:.6f}",
    f"Novelty threshold: {novelty_threshold:.6f}",
    "",
    "Held-out test metrics:",
    test_metrics.round(4).to_string(),
    "",
    "Held-out selective prediction:",
    pd.DataFrame([test_selective]).round(4).to_string(index=False),
    "",
    "Limitations:",
    "- Novelty scores are heuristic warnings, not proof of distribution shift.",
    "- Confidence does not guarantee correctness.",
    "- Lower selective error may come at the cost of lower coverage.",
    "- This notebook is not a clinical decision tool."
]
report_text = "\n".join(report_lines)
print(report_text)
with open("oracle_zero_audit_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)
test_decisions.to_csv("oracle_zero_test_decisions.csv", index=False)
print("\nSaved oracle_zero_audit_report.txt and oracle_zero_test_decisions.csv")

## 15. Suggested extensions

1. Repeat experiments across datasets and random seeds.
2. Add group-based validation for repeated entities and time-based validation for temporal data.
3. Compare multiple uncertainty and out-of-distribution methods.
4. Report confidence intervals for risk–coverage measurements.
5. Add unit tests using controlled synthetic examples.
6. Record dataset versions, package versions, and experiment configuration.
7. Build a FastAPI backend and Streamlit or React interface after the analysis engine is validated.

### Suggested GitHub description

> ORACLE ZERO is an experimental machine-learning reliability platform combining probability calibration, input-novelty detection, selective prediction, stress testing, and audit reports to investigate when a model should predict and when it should abstain.